***

# **Rail Track with Performance Restrictions by Region**

***

1) From the NTD Transit Way Mileage inventory, for each year, agency, and mode combination, get the miles of rail track with performance restrictions (element name = 'Track Under Performance Restriction')
2) From the same inventory, for each year, agency, and mode combination, get the miles of rail track regardless of presence of performance restrictions
3) From the same inventory, for each year, agency, and mode combination, get the percent of rail track with performance restrictions
4) Sum all those figures, but summed up for all agencies, and all modes 

Refer to the data dictionary [here](https://catalog.dvrpc.org/dataset/transit-conditions/resource/484a2739-5ce9-4913-bbc3-e60df5402554) and scroll down to see it.  

In [1]:
# Packages

import pandas as pd
import os
import itertools

pd.set_option('display.max_columns', None)

In [2]:
# User paths

user = os.getlogin()
path_users = os.path.join('C:\\Users', user)
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_transit = os.path.join(path_sp, 'Products', 'RHNA', 'New Data Collected', 'Transit Conditions')
path_data = os.path.join(path_transit, 'Transit Conditions Data Sources.xlsx')

In [ ]:
# Loading the data
sources = pd.read_excel(path_data)

# Getting the unique IDs NTD that SACOG previously used 
path_ids = os.path.join(path_transit, 'Copy of sacramento_area_council_of_governments.xlsx')
data_ids = pd.read_excel(path_ids)
ids = data_ids['NTD ID'].unique().tolist()

# Filter for 'NTD Transit Way Mileage'
mileage_data = sources[sources["Name"] == "NTD Transit Way Mileage"]

In [48]:

# Filter for 'NTD Facility Inventory'
facility_data = sources[sources["Name"] == "NTD Transit Way Mileage"]

all_data = []

# Using the sources file, we just loop through the URLs we care about rather than directly having to download the data
for index, row in facility_data.iterrows():
    url = row["URL"]
    year = row["Year"]
    
    try:
        data_excel = pd.read_excel(url)
        data_excel["Year"] = year  # Add the Year column
        all_data.append(data_excel)
        
        print(f"Successfully loaded the data for {year} from {url}")
    except Exception as e:
        print(f"Failed to read {url} for {year}: {e}")

data = pd.concat(all_data, ignore_index=True)

# Filter by agency now, used the same ones as in here: https://ntd-monthly-ridership--cal-itp-data-analyses.netlify.app/rtpa_sacramento-area-council-of-governments/00__monthly_ridership_report__rtpa_sacramento-area-council-of-governments
agency = ['Sacramento Regional Transit District', 'University of California, Davis',
                'Yolo County Transportation District', 'Yuba-Sutter Transit Authority',
                'City of Elk Grove', 'Paratransit, Inc.', 'City of Folsom',
                'County of Sacramento Municipal Services Agency', 'Paratransit, Inc. CTSA'
                ]

data = data[data['NTD ID'].isin(ids)]
# data = data[data['NTD ID'].isin(agency)] #also works

Successfully loaded the data for 2018 from https://www.transit.dot.gov/sites/fta.dot.gov/files/2018%20Transit%20Way%20Mileage.xlsx
Successfully loaded the data for 2019 from https://www.transit.dot.gov/sites/fta.dot.gov/files/2020-10/2019%20Transit%20Way%20Mileage.xlsx
Successfully loaded the data for 2020 from https://www.transit.dot.gov/sites/fta.dot.gov/files/2021-11/2020%20Transit%20Way%20Mileage.xlsx
Successfully loaded the data for 2021 from https://www.transit.dot.gov/sites/fta.dot.gov/files/2022-10/2021%20Transit%20Way%20Mileage.xlsx
Successfully loaded the data for 2022 from https://www.transit.dot.gov/sites/fta.dot.gov/files/2024-01/2022%20Transit%20Way%20Mileage.xlsx
Successfully loaded the data for 2023 from https://www.transit.dot.gov/sites/fta.dot.gov/files/2024-10/2023%20Transit%20Way%20Mileage.xlsx


In [51]:
# Main frame formatting
data.columns = data.columns.str.strip()

data = data[['Year', 'Agency Name', 'Mode', 'Element Name', 'Track Miles', 'NTD ID']].copy()
data['Track Miles'] = pd.to_numeric(data['Track Miles'], errors='coerce')

# Get the miles of rail tracks under performance metrics
restricted_miles = (
    data[data['Element Name'] == 'Track Under Performance Restriction']
    .groupby(['Year', 'Agency Name', 'Mode', 'NTD ID'])['Track Miles']
    .sum()
    .reset_index()
    .rename(columns={'Track Miles': 'Restricted Track Miles'})
)

# Get the total miles regardless of performance
total_miles = (
    data.groupby(['Year', 'Agency Name', 'Mode', 'NTD ID'])['Track Miles']
    .sum()
    .reset_index()
    .rename(columns={'Track Miles': 'Total Track Miles'})
)

# Merge and calc percents
merged_data = pd.merge(total_miles, restricted_miles, on=['Year', 'Agency Name', 'Mode', 'NTD ID'], how='left')
merged_data['Restricted Track Miles'] = merged_data['Restricted Track Miles'].fillna(0)
merged_data['Percent Restricted'] = (merged_data['Restricted Track Miles'] / merged_data['Total Track Miles']) * 100

merged_data["mode_name"] = merged_data["Mode"].map({"LR": "Light Rail", "CR": "Commuter Rail", "HR": "Heavy Rail", "SR": "Streetcar", "YR": "Hybrid Rail"})

# To match DVRPC
merged_data.rename(columns={
    "Year": "year",
    "NTD ID": "ntd_id",
    "Agency Name": "agency_name",
    "Mode": "mode",
    "Total Track Miles": "rt_mi",
    "Restricted Track Miles": "rt_w_prs",
    "Percent Restricted": "pct_w_prs"
}, inplace=True)

merged_data = merged_data[['year', 'ntd_id', 'agency_name', 'mode', 'mode_name', 'rt_mi', 'rt_w_prs', 'pct_w_prs']]

display(merged_data.head(6))

,year,ntd_id,agency_name,mode,mode_name,rt_mi,rt_w_prs,pct_w_prs
0,2018,90019,Sacramento Regional Transit District,LR,Light Rail,171.89,4.09,2.379429
1,2019,90019,Sacramento Regional Transit District,LR,Light Rail,172.80,5.00,2.893519
2,2020,90019,Sacramento Regional Transit District,LR,Light Rail,168.32,0.52,0.308935
3,2021,90019,Sacramento Regional Transit District,LR,Light Rail,168.54,0.74,0.439065
4,2022,90019,Sacramento Regional Transit District,LR,Light Rail,171.57,3.77,2.197354
5,2023,90019,Sacramento Regional Transit District,LR,Light Rail,170.76,2.96,1.733427


In [53]:
# Exports

# Percent Def by County

path_rail = os.path.join(path_transit, "Rail Track with Performance Restrictions")

print(''); print('')
print('Converting/exporting Rail Track Performance results to SharePoint')

# Export bridge cond
filename = f"Rail Track with Performance Restrictions - SACOG"
name_out_csv = filename + '.csv'
path_out_csv = os.path.join(path_rail, name_out_csv)
merged_data.to_csv(path_out_csv, index = False)

# print(f"Excel files exported here:  {path_out_xlsx}");print('')
print(f"Files exported here:  {path_transit}");print('')



Converting/exporting Rail Track Performance results to SharePoint
Files exported here:  C:\Users\jchoy\Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents\Products\RHNA\New Data Collected\Transit Conditions

